# GLM làm nền cho cây

Bước 12 của kế hoạch S6E11.

GLM cung cấp logit ban đầu. Cây học phần hiệu chỉnh trên logit rồi chuyển tổng về xác suất. Quy trình chọn số cây giữ riêng tập dừng sớm, sau đó huấn luyện lại trên toàn bộ phần huấn luyện của fold.

## Chạy lại

Chạy từ thư mục `plan_runs` với môi trường Python đã dùng cho RTX 4060. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```powershell
& '..\.build\venv\Scripts\python.exe' -B -X utf8 run_plan.py --stage advanced --compare-xgb-digits --tune-family xgb --seed-anchor XGB-D03
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'advanced', '--compare-xgb-digits', '--tune-family', 'xgb', '--seed-anchor', 'XGB-D03']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `GLM-BOOST-LGB` | `T02` | 0.948623 | +0.000097 | 0.522113 | 0.044872 | 396.8 | 0.947683 | 0.950164 | 0.953800 | 0.948633 | 0.943055 |
| `GLM-BOOST-XGB` | `T03` | 0.948544 | +0.000050 | 0.521212 | 0.044924 | 153.6 | 0.947633 | 0.950265 | 0.953487 | 0.948543 | 0.942984 |

## Đọc dữ liệu kết quả hiện tại

Cell này xác minh những lượt hoàn tất từ dự đoán OOF và tệp metric.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import final_summary, read_stage

data_dir = plan_dir / "results" / "full"
rows, pending, notes = read_stage('05-glm-boost', data_dir)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if False:
    display(Markdown(final_summary(data_dir, markdown_links=False)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    directory = data_dir / record["name"]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    with np.load(directory / "predictions.npz", allow_pickle=False) as saved:
        display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

Các tệp dự đoán đủ để tính lại AUC. Khả năng nạp lại trọn bộ GLM, tiền xử lý và cây cần kiểm tra bằng model bundle riêng.